In [21]:
import asyncio
import os
import shutil
from datetime import datetime
from playwright.async_api import async_playwright
from feedgen.feed import FeedGenerator

# --- 【パス設定】絶対パスで固定 ---
BASE_DIR = "/home/yoshikazu-obikawa/dev/seikyoRSS"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
RSS_FILE = os.path.join(BASE_DIR, "seikyo_news.xml")

# --- 設定情報 ---
USER_ID = "cyanobi.2.29@gmail.com"
PASSWORD = "B0eceJ*kz%"
# ※Funnelを使用する場合はここをFunnelのURLに変更してください
GITHUB_BASE_URL = "https://cyanobi.github.io/seikyoRSS/"

CATEGORIES = {
    "報道・連載": "https://www.seikyoonline.com/news/",
    "池田大作先生": "https://www.seikyoonline.com/president/",
    "体験・教学": "https://www.seikyoonline.com/experience_kyougaku/",
    "生活・文化": "https://www.seikyoonline.com/lifestyle/",
    "投稿": "https://www.seikyoonline.com/toukou/",
    "漫画": "https://www.seikyoonline.com/comic/",
    "デジタル特集": "https://www.seikyoonline.com/digital/",
    "ユース特集": "https://www.seikyoonline.com/youth/",
    "大白蓮華": "https://www.seikyoonline.com/add_contents/daibyakurenge/"
}

now = datetime.now()
TARGET_DATE = f"{now.year}年{now.month}月{now.day}日"

async def scrape_category(page, category_name, url, fg):
    print(f"📂 [巡回] {category_name} にアクセス中...")
    try:
        await page.goto(url, wait_until="networkidle", timeout=60000)
        # Lazy Load対策：画像を読み込ませるために深めにスクロール
        await page.evaluate("window.scrollBy(0, 2000)")
        await asyncio.sleep(4) 

        # 取得対象のセレクタ（最新の構造に追従）
        blocks = await page.query_selector_all("div.p2o_text, div.p2o_text_photo, .news_list_block, .article-item, .list_item, div.p2ophoto")

        local_count = 0
        for block in blocks:
            link_el = await block.query_selector("a[href*='article']")
            if not link_el: continue
            raw_href = await link_el.get_attribute("href")
            
            # 日付判定
            date_el = await block.query_selector(".ts_days, .date, .p2o_date")
            date_text = await date_el.inner_text() if date_el else await block.inner_text()
            if TARGET_DATE not in date_text: continue

            # タイトル取得
            title_el = await block.query_selector(".under, h3, .shosai-title, .title")
            title = await title_el.inner_text() if title_el else "タイトル不明"

            # --- 【重要】画像処理（Referer対応版） ---
            img_el = await block.query_selector("img")
            final_img_url = None
            
            if img_el:
                # src, data-src 両方チェック
                src_url = await img_el.get_attribute("src") or await img_el.get_attribute("data-src")
                
                if src_url and "common" not in src_url and "spacer" not in src_url:
                    # URL補完
                    if src_url.startswith("//"): src_url = f"https:{src_url}"
                    elif src_url.startswith("/"): src_url = f"https://www.seikyoonline.com{src_url}"

                    try:
                        img_name = src_url.split("/")[-1].split("?")[0]
                        if not img_name.endswith((".jpg", ".png", ".jpeg")): img_name += ".jpg"
                        img_path = os.path.join(IMAGE_DIR, img_name)
                        
                        # 【核心】Refererを添えてブラウザセッションで取得
                        img_res = await page.request.get(src_url, headers={
                            "Referer": "https://www.seikyoonline.com/"
                        })
                        
                        if img_res.status == 200:
                            with open(img_path, "wb") as f:
                                f.write(await img_res.body())
                            if os.path.exists(img_path) and os.path.getsize(img_path) > 0:
                                final_img_url = f"{GITHUB_BASE_URL}images/{img_name}"
                    except: pass

            # リンクの正規化
            full_url = f"https:{raw_href}" if raw_href.startswith("//") else raw_href
            if not full_url.startswith("http"): full_url = f"https://www.seikyoonline.com{raw_href}"

            # RSSエントリ追加
            fe = fg.add_entry()
            fe.title(f"[{category_name}] {title.strip()}")
            fe.link(href=full_url)
            fe.id(full_url)
            fe.pubDate(datetime.now().astimezone()) # 更新日時を付与
            
            desc_text = f"カテゴリ: {category_name} / 公開日: {date_text.strip()}"
            if final_img_url:
                fe.description(f'<img src="{final_img_url}" style="max-width:100%;"><br>{desc_text}')
                fe.enclosure(final_img_url, 0, 'image/jpeg')
            else:
                fe.description(desc_text)
            
            local_count += 1
            print(f"  [+] {title.strip()[:15]}... {'📸' if final_img_url else '📄'}")

        return local_count
    except Exception as e:
        print(f"  ⚠️ {category_name} エラー: {e}")
        return 0

async def main():
    # 画像フォルダのリセット
    if os.path.exists(IMAGE_DIR):
        shutil.rmtree(IMAGE_DIR)
    os.makedirs(IMAGE_DIR, exist_ok=True)

    async with async_playwright() as p:
        print(f"\n🚀 [1/5] ブラウザ起動...")
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': 1280, 'height': 1200},
            locale="ja-JP",
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        try:
            print(f"🔑 [2/5] ログイン実行...")
            await page.goto("https://www.seikyoonline.com/auth/login", wait_until="networkidle")
            await page.fill("input[placeholder*='SOKA ID']", USER_ID)
            await page.fill("input[placeholder*='パスワード']", PASSWORD)
            await page.click("button:has-text('ログイン')")
            
            # ログイン後の遷移を確実に待つ
            await page.wait_for_load_state("networkidle")
            await asyncio.sleep(6) 
            
            fg = FeedGenerator()
            fg.title(f"聖教新聞 RSSフィード")
            fg.link(href=GITHUB_BASE_URL)
            fg.description(f"本日の聖教新聞ニュースまとめ ({TARGET_DATE})")
            fg.language('ja')

            print(f"🔄 [3/5] カテゴリ巡回開始（ターゲット: {TARGET_DATE}）")
            total_count = 0
            for name, url in CATEGORIES.items():
                total_count += await scrape_category(page, name, url, fg)

            print(f"\n📊 [4/5] 収集完了: {total_count} 件")

            if total_count > 0:
                print(f"💾 [5/5] RSS保存...")
                fg.rss_file(RSS_FILE)
                print(f"✨ 完了: images/ と {RSS_FILE} を更新しました。")
            else:
                print("⚠️ 該当記事がありません。日付や構造を確認してください。")

        except Exception as e:
            print(f"❌ メイン処理エラー: {e}")
        finally:
            await browser.close()

if __name__ == "__main__":
    asyncio.run(main())


🚀 [1/5] ブラウザ起動...
🔑 [2/5] ログイン実行...
🔄 [3/5] カテゴリ巡回開始（ターゲット: 2026年4月25日）
📂 [巡回] 報道・連載 にアクセス中...
  [+] アメリカ創価大学・地球的問題群... 📄
  [+] 自信がなくてもいい。「不完全で... 📄
  [+] 〈第10回本部幹部会〉　原田稔... 📄
  [+] 〈社説〉　2026・4・25　... 📄
  [+] 月々日々に――池田先生の折々の... 📄
  [+] 名字の言　目には見えない大切な... 📄
  [+] 寸鉄
2026年4月25日... 📄
  [+] 【茨城】新たな広布の扉をわれら... 📄
📂 [巡回] 池田大作先生 にアクセス中...


Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


📂 [巡回] 体験・教学 にアクセス中...
  [+] 〈Seikyo　Gift〉　書... 📄
  [+] 〈きょうの発心〉　衣食御書　広... 📄
📂 [巡回] 生活・文化 にアクセス中...
  [+] 【電子版限定】〈文化〉　ワクワ... 📄
📂 [巡回] 投稿 にアクセス中...
  [+] 〈声〉　「安心して先祖を託せる... 📄
📂 [巡回] 漫画 にアクセス中...
📂 [巡回] デジタル特集 にアクセス中...
  [+] 「一歩」を踏み出すこと　〈日め... 📄
📂 [巡回] ユース特集 にアクセス中...
  [+] 「KIBOUロゴ」ぬり絵　応募... 📄
📂 [巡回] 大白蓮華 にアクセス中...

📊 [4/5] 収集完了: 14 件
💾 [5/5] RSS保存...
✨ 完了: images/ と /home/yoshikazu-obikawa/dev/seikyoRSS/seikyo_news.xml を更新しました。
